In [1]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import torch.nn as nn
import torch.optim as optim

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


In [2]:
df = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_train.csv'))
display(df.head())

,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Demand_Response_Capacity_kW
0,siteA,2019-01-01 00:00:00,22.20,0.0,4.8,0,0.0
1,siteA,2019-01-01 00:15:00,22.27,0.0,4.8,0,0.0
2,siteA,2019-01-01 00:30:00,22.35,0.0,4.8,0,0.0
3,siteA,2019-01-01 00:45:00,22.42,0.0,4.8,0,0.0
4,siteA,2019-01-01 01:00:00,22.50,0.0,4.8,0,0.0


In [3]:
df.groupby("Demand_Response_Flag")['Site'].count()

Demand_Response_Flag
-1      2262
 0    102011
 1       847
Name: Site, dtype: int64

In [4]:
def preprocess_data(df_in):
    df = df_in.copy()
    # Convert 'Timestamp' to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp_Local'])
    # Extract datetime features
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
    df['Minute'] = df['Timestamp'].dt.minute
    # Build seasonal features
    df['Is_Weekend'] = df['Weekday'].isin([5, 6]).astype(int)
    df['Is_Summer'] = df['Month'].isin([5, 6, 7, 8]).astype(int)
    df['Is_Winter'] = df['Month'].isin([12, 1, 2, 3]).astype(int)
    # Create hour of day categories
    df['Is_Afternoon'] = df['Hour'].isin(range(12, 18)).astype(int)
    df['Is_Evening'] = df['Hour'].isin(range(18, 24)).astype(int)
    # drop unused columns
    df.drop(columns=['Timestamp_Local','Timestamp','Site','Demand_Response_Capacity_kW'], inplace=True)
    # Fix target variable (instead of -1 make it 2)
    df['Demand_Response_Flag'] = df['Demand_Response_Flag'].replace(-1, 2)
    return df

def standardize_data(X, subset=None):
    scaler = StandardScaler()
    if subset is not None:
        X_subset = X[subset]
        X_scaled_subset = scaler.fit_transform(X_subset)
        X_scaled = X.copy()
        X_scaled[subset] = X_scaled_subset
    else:
        X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler

df = preprocess_data(df)

In [5]:
display(df.head(), df.tail())

,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Hour,Day,Month,Weekday,Minute,Is_Weekend,Is_Summer,Is_Winter,Is_Afternoon,Is_Evening
0,22.20,0.0,4.8,0,0,1,1,1,0,0,0,1,0,0
1,22.27,0.0,4.8,0,0,1,1,1,15,0,0,1,0,0
2,22.35,0.0,4.8,0,0,1,1,1,30,0,0,1,0,0
3,22.42,0.0,4.8,0,0,1,1,1,45,0,0,1,0,0
4,22.50,0.0,4.8,0,1,1,1,1,0,0,0,1,0,0


,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Hour,Day,Month,Weekday,Minute,Is_Weekend,Is_Summer,Is_Winter,Is_Afternoon,Is_Evening
105115,20.57,0.0,56.33,0,22,31,12,6,45,1,0,1,0,1
105116,20.60,0.0,56.33,0,23,31,12,6,0,1,0,1,0,1
105117,20.75,0.0,56.33,0,23,31,12,6,15,1,0,1,0,1
105118,20.90,0.0,56.33,0,23,31,12,6,30,1,0,1,0,1
105119,21.05,0.0,56.33,0,23,31,12,6,45,1,0,1,0,1


In [ ]:
# Some classes and functions for training

# Define the neural network architecture: Just need to load architecture from net_architecture.py
# see the file for details
from net_architecture import Net

# Define Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        # calculate focal loss (using cross-entropy loss)
        # but this form of focal loss is more numerically stable
        # What it does: FocusLoss suppress majority class loss, focus more on minority class loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


# prepare data for training (train_loader, test_loader)
def prepare_data(X_train, y_train, X_test, y_test, batch_size=1024):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)
    # create datasets and dataloaders
    #------------------------------------------------------
    train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    #------------------------------------------------------
    test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    #------------------------------------------------------
    return train_loader, test_loader


def generate_criterion(y_train, loss_type='cross_entropy'):
    # 1. Calculate class weights to handle class imbalance
    class_weights_tensor = torch.tensor(
        compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train), 
        dtype=torch.float32
    )
    # 2. Choose which loss function to use
    if loss_type == 'cross_entropy':
        # Standard CrossEntropyLoss with class weights
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    elif loss_type == 'focal':
        # Use class_weights_tensor as alpha for Focal Loss
        criterion = FocalLoss(alpha=class_weights_tensor, gamma=2)
    return criterion


# training loop
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs=50):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        # Evaluate on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
            test_loss /= len(test_loader.dataset)
        if (epoch + 1) % 10 == 0:
            avg_loss = running_loss / len(train_loader.dataset)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Test Loss: {test_loss:.4f}")
        model.train()
    return model


# evaluation function
def evaluate_model(model, data_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in data_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    check_acc = torch.tensor(all_preds) == torch.tensor(all_labels)
    accuracy = (check_acc).float().mean().item()
    print(f"Test Accuracy: {accuracy:.4f}")
    f1 = f1_score(all_labels, all_preds, average='weighted')
    print(f"Weighted F1 Score: {f1:.4f}")

In [7]:
# Assume the target column is 'Demand_Response_Flag' (3 classes: 0, 1, 2)
# If not, replace with the correct target column

# Prepare features and target
X = df.drop(columns=['Demand_Response_Flag']).values
y = df['Demand_Response_Flag'].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.1, random_state=42, stratify=y
)

# Preparations for trainings
input_dim = X_train.shape[1]
num_classes = 3
n_epochs = 100
batch_sz = 1024
lr_ = 0.001
w_decay = 1e-5

# configure data loader and batch size
# 1. convert data to torch tensors
# 2. create datasets and dataloaders
train_loader, test_loader = prepare_data(X_train, y_train, X_test, y_test, batch_size=batch_sz)

# create a neural net model
model = Net(input_dim, num_classes)

# generate criterion (loss function)
# for imbalanced dataset, recommend to use focal loss
criterion = generate_criterion(y_train, loss_type='focal')

# configure optimizer
optimizer = optim.Adam(model.parameters(), lr=lr_, weight_decay=w_decay)

# Train the model
model = train_model(model, train_loader, test_loader, criterion, optimizer, epochs=n_epochs)

# Evaluate the model
evaluate_model(model, test_loader)

Epoch 10/100, Loss: 0.1742, Test Loss: 0.1455
Epoch 20/100, Loss: 0.0996, Test Loss: 0.0828
Epoch 30/100, Loss: 0.0702, Test Loss: 0.0555
Epoch 40/100, Loss: 0.0581, Test Loss: 0.0391
Epoch 50/100, Loss: 0.0507, Test Loss: 0.0326
Epoch 60/100, Loss: 0.0461, Test Loss: 0.0317
Epoch 70/100, Loss: 0.0396, Test Loss: 0.0289
Epoch 80/100, Loss: 0.0368, Test Loss: 0.0277
Epoch 90/100, Loss: 0.0322, Test Loss: 0.0301
Epoch 100/100, Loss: 0.0305, Test Loss: 0.0192
Test Accuracy: 0.9307
Weighted F1 Score: 0.9484


In [8]:
# Save the scaler to a pkl file
joblib.dump(scaler, './models/phase1_scaler.pkl')
# Save the trained model to a file
torch.save(model.state_dict(), './models/phase1_nn_model.pth')

# Load the trained model from file
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('./models/phase1_nn_model.pth'))
model_loaded.eval()

Net(
  (fc1): Linear(in_features=13, out_features=256, bias=True)
  (relu): ReLU()
  (gelu): GELU(approximate='none')
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=128, bias=True)
  (fc5): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [9]:
# Generate predictions and calculate distribution of predicted classes
model.eval()
predicted = []
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        predicted.extend(preds.cpu().numpy())
predicted = torch.tensor(predicted)
unique_pred, counts_pred = torch.unique(predicted, return_counts=True)
distribution_pred = dict(zip(unique_pred.tolist(), counts_pred.tolist()))
print("Distribution in predicted:", distribution_pred)

Distribution in predicted: {0: 9472, 1: 385, 2: 655}
